# TP 2 — Partie 2 : nettoyer le texte et découper le JSON en couche Silver

**Objectif :** obtenir des tables Silver propres à partir des deux sources de la partie 1 :

| Table Silver | Une ligne par | Source |
|---|---|---|
| `silver.avis` | avis d'une commande | `bronze.avis` |
| `silver.suivi_colis` | colis | `bronze.suivi_brut` |
| `silver.suivi_evenements` | étape du suivi d'un colis | `bronze.suivi_brut` |

Les règles du TP 1 s'appliquent toujours : types explicites, pas de doublons sur la clé, lignes invalides mises en **quarantaine**.

> **⚠️ Un seul notebook actif à la fois.** Arrêtez le noyau des autres notebooks avant de commencer.

In [ ]:
from pyspark.sql import SparkSession, functions as F, Window

spark = SparkSession.builder.appName("tp2-silver").getOrCreate()


def a_completer(quoi):
    """Signale clairement une partie du TP qui n'a pas encore été écrite."""
    raise NotImplementedError(f"⚠️ Cellule à compléter : {quoi}")


def doublons(df, cle):
    return df.count() - df.dropDuplicates(cle).count()

## Étape 1 — Trouver la clé des avis

On s'attend à ce que `review_id` identifie un avis. Vérifions.

In [ ]:
avis_bronze = spark.table("lakehouse.bronze.avis")
print("lignes                           :", avis_bronze.count())
print("doublons sur review_id           :", doublons(avis_bronze, ["review_id"]))
print("doublons sur (review_id, order_id):", doublons(avis_bronze, ["review_id", "order_id"]))
print("commandes avec plusieurs avis    :", doublons(avis_bronze, ["order_id"]))

exemple = (avis_bronze.groupBy("review_id").count().filter("count > 1").first()["review_id"])
avis_bronze.filter(F.col("review_id") == exemple) \
    .select("review_id", "order_id", "review_score", "review_creation_date").show(truncate=False)

> **❓ Question 1.** Les lignes qui partagent un même `review_id` sont-elles de vrais doublons à supprimer ? Quelle est la bonne clé de `silver.avis` ? Que se passerait-il si l'on gardait une seule ligne par `review_id` ?

## Étape 2 — Normaliser le texte

Pour analyser du texte, on le **normalise** : tout en minuscules, sans accents, sans ponctuation ni chiffres, avec des espaces simples. *« Ótimo produto!! Chegou rápido. »* devient *« otimo produto chegou rapido »*.

La fonction `normaliser` est fournie. Testez-la.

In [ ]:
ACCENTS, SANS_ACCENTS = "áàâãäéèêëíìîïóòôõöúùûüçñ", "aaaaaeeeeiiiiooooouuuucn"


def normaliser(colonne):
    texte = F.lower(colonne)
    texte = F.translate(texte, ACCENTS, SANS_ACCENTS)
    texte = F.regexp_replace(texte, r"[^a-z\s]", " ")      # ponctuation, chiffres, émojis
    texte = F.regexp_replace(texte, r"\s+", " ")           # espaces et retours à la ligne
    return F.trim(texte)


spark.createDataFrame([("Ótimo produto!! Chegou rápido.\nRecomendo 100%",)], ["t"]) \
    .select(normaliser(F.col("t")).alias("normalise")).show(truncate=False)

## Étape 3 — Créer `silver.avis` et sa quarantaine

Créez un DataFrame `avis` qui :
1. **type** les colonnes : `review_score` en entier, `review_creation_date` et `review_answer_timestamp` en `timestamp` ;
2. regroupe le titre et le message dans `commentaire` : `F.concat_ws(" ", "review_comment_title", "review_comment_message")`, puis `F.trim`, et `null` si le résultat est vide ;
3. ajoute `commentaire_normalise` (fonction `normaliser`), `nb_mots` (`F.size(F.split("commentaire_normalise", " "))`, `0` s'il n'y a pas de commentaire) et `est_negatif` (`true` si la note est 1 ou 2) ;
4. garde une seule ligne par clé `(review_id, order_id)`.

Puis mettez en **quarantaine** les avis dont la commande n'existe pas dans `silver.orders` (anti-jointure, comme au TP 1).

In [ ]:
commandes = spark.table("lakehouse.silver.orders").select("order_id")

avis = a_completer("avis")  # À COMPLÉTER

orphelins = avis.join(commandes, "order_id", "left_anti")
orphelins.writeTo("lakehouse.silver.quarantaine_avis").using("iceberg").createOrReplace()
avis.join(commandes, "order_id", "left_semi") \
    .writeTo("lakehouse.silver.avis").using("iceberg").createOrReplace()

print("silver.avis      :", spark.table("lakehouse.silver.avis").count())
print("quarantaine_avis :", spark.table("lakehouse.silver.quarantaine_avis").count())
spark.table("lakehouse.silver.avis").filter("nb_mots > 0") \
    .select("review_score", "commentaire", "commentaire_normalise").show(5, truncate=45)

> **❓ Question 2.** Pourquoi certains avis sont-ils « orphelins » ? (Pensez à la quarantaine des commandes du TP 1.) Quelle part des avis négatifs contient un commentaire, contre quelle part des avis positifs ?

In [ ]:
# À COMPLÉTER : part des avis avec commentaire, selon est_negatif

## Étape 4 — Lire le JSON avec un schéma explicite

En Silver, on ne laisse pas Spark deviner : le schéma est **écrit**, et il couvre les deux versions du format (les champs de la version 2 sont simplement `null` dans les messages de 2017).

`F.from_json` transforme le texte en structure. Un message tronqué donne `null` : il part en quarantaine.

In [ ]:
SCHEMA_SUIVI = """
    colis_id STRING,
    order_id STRING,
    emis_le STRING,
    version_schema INT,
    transporteur STRUCT<nom: STRING, service: STRING>,
    evenements ARRAY<STRUCT<code: STRING, horodatage: STRING,
                            lieu: STRUCT<ville: STRING, etat: STRING>>>,
    incidents ARRAY<STRUCT<type: STRING, horodatage: STRING>>
"""

messages = (spark.table("lakehouse.bronze.suivi_brut")
    .withColumn("m", F.from_json("value", SCHEMA_SUIVI)))

(messages.filter(F.col("m.colis_id").isNull())
    .select("value", "_source", "_ingere_le")
    .writeTo("lakehouse.silver.quarantaine_suivi").using("iceberg").createOrReplace())

valides = messages.filter(F.col("m.colis_id").isNotNull()).select("m.*")
valides.printSchema()
print("messages valides :", valides.count(),
      "| en quarantaine :", spark.table("lakehouse.silver.quarantaine_suivi").count())

## Étape 5 — Créer `silver.suivi_colis`

Une ligne par colis. Créez un DataFrame `colis` à partir de `valides` qui :
1. supprime les messages **renvoyés deux fois** (une seule ligne par `colis_id`) ;
2. aplatit les objets : `transporteur.nom` → `transporteur`, `transporteur.service` → `service` ;
3. remplace une `version_schema` absente par `1` et convertit `emis_le` en `timestamp` ;
4. ajoute `nb_evenements` et `nb_incidents`, le nombre d'éléments des listes.

**Attention :** sur une liste `null`, `F.size` renvoie `-1`, pas `0`. Pour les messages de 2017, `nb_incidents` doit valoir `0`.

In [ ]:
print("messages renvoyés deux fois :", doublons(valides, ["colis_id"]))

colis = a_completer("colis")  # À COMPLÉTER

colis.writeTo("lakehouse.silver.suivi_colis").using("iceberg").createOrReplace()
spark.table("lakehouse.silver.suivi_colis").groupBy("version_schema", "nb_incidents").count() \
    .orderBy("version_schema", "nb_incidents").show()

## Étape 6 — Créer `silver.suivi_evenements`

Une ligne par étape du suivi. `F.explode` transforme une liste en autant de lignes qu'elle a d'éléments : un colis à 5 événements donne 5 lignes.

Créez un DataFrame `evenements` à partir des colis dédoublonnés : `colis_id`, `order_id`, puis les champs de chaque événement (`code`, `horodatage` en `timestamp`, `ville`, `etat`).

In [ ]:
uniques = valides.dropDuplicates(["colis_id"])

evenements = a_completer("evenements")  # À COMPLÉTER

evenements.writeTo("lakehouse.silver.suivi_evenements").using("iceberg").createOrReplace()
spark.table("lakehouse.silver.suivi_evenements").groupBy("code").count().orderBy(F.desc("count")).show()

## Étape 7 — La durée de transit

Pour chaque colis, la durée de transit est le temps entre la prise en charge (`PRIS_EN_CHARGE`) et la livraison (`LIVRE`). On la calcule à partir de `silver.suivi_evenements`, puis on l'ajoute à `silver.suivi_colis`.

In [ ]:
duree = a_completer("duree : une ligne par colis_id avec duree_transit_h")  # À COMPLÉTER

(colis.join(duree, "colis_id", "left")
    .writeTo("lakehouse.silver.suivi_colis").using("iceberg").createOrReplace())
spark.table("lakehouse.silver.suivi_colis").select("duree_transit_h").summary("count", "mean", "50%", "max").show()

> **❓ Question 3.** Certains messages listent leurs événements dans le désordre (le transporteur les envoie dans l'ordre où il les reçoit). Votre calcul de durée en dépend-il ? Pourquoi certains colis n'ont-ils pas de durée ?

## Bilan

In [ ]:
bilan = [
    ("avis", spark.table("lakehouse.bronze.avis").count(), spark.table("lakehouse.silver.avis").count()),
    ("suivi (messages → colis)", spark.table("lakehouse.bronze.suivi_brut").count(),
     spark.table("lakehouse.silver.suivi_colis").count()),
]
spark.createDataFrame(bilan, ["source", "bronze", "silver"]).withColumn("ecart", F.col("bronze") - F.col("silver")).show(truncate=False)
spark.sql("SHOW TABLES IN lakehouse.silver").show()

> **❓ Question 4.** Expliquez chaque écart entre Bronze et Silver par une règle : combien de lignes, pour quelle raison ?